In [1]:
import micropip
await micropip.install("tabulate")
await micropip.install("pandas")

import math
import pandas as pd

In [2]:
MICROSECOND = 1
MILLISECOND = 1000 * MICROSECOND
SECOND      = 1000 * MILLISECOND
MINUTE      = 60 * SECOND
HOUR        = 60 * MINUTE
DAY         = 24 * HOUR
MONTH       = 30 * DAY # Approximation as months are irregular
YEAR        = 365 * DAY # Defining YEAR in terms of days is a better approximation. 12 months would give 360 days. Note however that some years (leap years) have 366 days.
CENTURY     = 100 * YEAR # Somewhat inacurate as it ignores leap years

TIMES = {
	"1 second": SECOND,
	"1 minute": MINUTE,
	"1 hour": HOUR,
	"1 day": DAY,
	"1 month": MONTH,
	"1 year": YEAR,
	"1 century": CENTURY,
}

In [39]:
def inverse_n_log_n(t: int) -> int:
	"""
	 Inverts the function n*lg(n)	

	 From https://math.stackexchange.com/a/1302746
	 n*lg(n) = t => n = t/lg(n)
	 We start with a guess for n set to t, run the 
	 guess through the rearranged function and check if we get n back.
	 If not, set n to the returned value.
	 Repeat the process until we get the same n back
	"""
	n = float(t)
	while True:
		next = t/math.log2(n)
		if int(next) == int(n):
			return int(next)
		n = next

In [40]:
def inverse_factorial(t: int) -> int:
	"""
	Inverts the factorial function

	If the result would not be an integer, it gives the next integer.
	e.g. 4! = 24, and inverseFactorial(24) = 4
	however inverseFactorial(25) = 5 even though 5! = 120
	"""
	n = 1
	r = t
	while r > 1:
		n+=1
		r/=n
	return n

In [41]:
FUNCTIONS = {
	"lg(n)":   lambda n: math.log2(n),
	"sqrt(n)": lambda n: math.sqrt(n),
	"n":       lambda n: n,
	"n*lg(n)": lambda n: n * math.log2(n),
	"n^2":     lambda n: n**2,
	"n^3":     lambda n: n**3,
	"2^n":     lambda n: 2**n,
	"n!":      lambda n: math.factorial(n)
}


INVERTS = {
	# "lg(n)":   lambda t: 2**t, # I took this out because the numbers get so large that pyodide was strugling to compute them
	"sqrt(n)": lambda t: t**2,
	"n":       lambda t: t,
	"n*lg(n)": lambda t: inverse_n_log_n(t),
	"n^2":     lambda t: math.sqrt(t),
	"n^3":     lambda t: math.cbrt(t),
	"2^n":     lambda t: math.log2(t),
	"n!":      lambda t: inverse_factorial(t)
}

In [42]:
grid = {}
for f_name, func in INVERTS.items():
	row = {}
	for t_name, time in TIMES.items():
		row[t_name] = f"{int(func(time)):e}"
	grid[f_name] = row

df = pd.DataFrame.from_dict(grid, orient="index")
print(df.to_markdown())

|         |   1 second |       1 minute |          1 hour |            1 day |         1 month |          1 year |        1 century |
|:--------|-----------:|---------------:|----------------:|-----------------:|----------------:|----------------:|-----------------:|
| sqrt(n) |      1e+12 |    3.6e+15     |     1.296e+19   |      7.46496e+21 |     6.71846e+24 |     9.94519e+26 |      9.94519e+30 |
| n       |      1e+06 |    6e+07       |     3.6e+09     |      8.64e+10    |     2.592e+12   |     3.1536e+13  |      3.1536e+15  |
| n*lg(n) |  62746     |    2.80142e+06 |     1.33378e+08 |      2.75515e+09 |     7.18709e+10 |     7.97634e+11 |      6.8611e+13  |
| n^2     |   1000     | 7745           | 60000           | 293938           |     1.60997e+06 |     5.61569e+06 |      5.61569e+07 |
| n^3     |    100     |  391           |  1532           |   4420           | 13736           | 31593           | 146645           |
| 2^n     |     19     |   25           |    31           |   